In [ ]:
import sys
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from matplotlib import cm
import matplotlib as mp
# import gaia_tools as gt
import scipy
from scipy.ndimage import gaussian_filter
import astropy.units as u
from astropy.coordinates import SkyCoord
import math
import h5py
# import healpy as hp
import pykdgrav3_utils
from pykdgrav3_utils import units
un = units.units(1, 600., verbose=True)
import healpy as hp
from healpy.newvisufunc import projview, newprojplot
from matplotlib.projections.geo import GeoAxes
# import plotly.graph_objects as go
from matplotlib import colors

sys.path.append('/home/dnurme/linux_env/Thesis/My_thesis/Modules/')
from mock_wake import generate_mock_wake
from rotation_funcs import rotate, angle_finder, rz, ry, rx
from misc import plot_OD_gaussian, overdensity, resultant_OD_gaussian 

# plot_OD_gaussian_interactive, plot_gaussian_interactive

In [ ]:
file1 = '/home/dnurme/linux_env/Data/dm_sim.00001.0'

def load_snap_file(path, part_type='PartType1', is_print = False):

    snap_file = h5py.File(path, 'r')
    part_data = snap_file[part_type]

    if(is_print):
        print(f'Loading snapshot: {path.split("/")[-1]}')
        print(f'Selected species: {part_type}')
        print(f'Snap file keys: {snap_file.keys()}')
        print(f'Part type keys: {part_data.keys()}')

    return part_data

In [ ]:
snap_stars1 = load_snap_file(file1, part_type='PartType4', is_print=True)
star_coord1 = snap_stars1['Coordinates'][:]*un.dKpcUnit

x1 = star_coord1[:,0]
y1 = star_coord1[:,1]
z1 = star_coord1[:,2]

In [ ]:
file = '/home/dnurme/linux_env/Data/pos_stars_Plummer_LMC.npy'
data = np.load(file)
x_plum = data[:,0]
y_plum = data[:,1]
z_plum = data[:,2]

star_coord2 = np.array([x_plum, y_plum, z_plum]).T

x2 = star_coord2[:,0]
y2 = star_coord2[:,1]
z2 = star_coord2[:,2]

In [ ]:
# print(star_coord1.shape)
# print(star_coord2.shape)


In [ ]:
resultant_OD_gaussian(star_coord1, star_coord2, (300, 300), 2, "Hernquist", "Plummer")

In [ ]:
from matplotlib.colors import Normalize

def resultant_OD_gaussian2(data1, data2, bins, sigma, data1_label, data2_label):
    
    def overdensity(x, y, bins):  # generating the overdensity map
        pre_OD, xedges, yedges = np.histogram2d(x, y, bins)
        OD = ((pre_OD / np.mean(pre_OD))-1)
        return OD, xedges, yedges
    
    x1, y1, z1 = data1[:, 0], data1[:, 1], data1[:, 2]
    x2, y2, z2 = data2[:, 0], data2[:, 1], data2[:, 2]

    OD1, xedges1, yedges1 = overdensity(x1, y1, bins)
    OD2, xedges2, yedges2 = overdensity(x2, y2, bins)

    hist_smoothed1 = gaussian_filter(OD1.T, sigma=sigma)
    hist_smoothed2 = gaussian_filter(OD2.T, sigma=sigma)

    resultant_OD = hist_smoothed1 - hist_smoothed2

    fig, axs = plt.subplots(1, 3, figsize=(24, 6), facecolor='white')
    im1 = axs[0].pcolormesh(xedges1, yedges1, hist_smoothed1,cmap="seismic", 
                           norm=Normalize(vmin=-0.3, 
                                          vmax=0.5))
    im2 = axs[1].pcolormesh(xedges2, yedges2, hist_smoothed2, cmap="seismic",
                          norm=Normalize(vmin=-np.max(np.abs(hist_smoothed2)), 
                                          vmax=np.max(np.abs(hist_smoothed2))))
    im3 = axs[2].pcolormesh(xedges1, yedges1, resultant_OD, cmap="seismic",
                          norm=Normalize(vmin=-np.max(np.abs(resultant_OD)), 
                                         vmax=np.max(np.abs(resultant_OD))))
    

    c1 = plt.colorbar(im1, ax=axs[0])
    c2 = plt.colorbar(im2, ax=axs[1])
    c3 = plt.colorbar(im3, ax=axs[2])

    for c in [c1, c2, c3]:
        c.ax.tick_params(labelsize=18)

    for ax in axs:
        ax.tick_params(axis='both', labelsize=18)
        ax.set_xlabel('x (kpc)', size=18)
        ax.set_xlim(-300, 100)
        ax.set_ylim(-200, 200)

    axs[0].set_ylabel('y (kpc)', size=18)

    axs[0].set_title(f'{data1_label} Profile', size=20)
    axs[1].set_title(f'{data2_label} Profile', size=20)
    axs[2].set_title('Resultant Overdensity', size=20)

    return fig, axs



resultant_OD_gaussian2(star_coord1, star_coord2, (300, 300), 2, "Hernquist", "Plummer")